# Módulo 1 (b) · Fundamentos de los LLM: embeddings, similitud y alucinaciones

**Duración sugerida:** 1.5 horas

## Objetivos del laboratorio

- Generar **embeddings** (vectores semánticos) con un modelo local.
- Medir **similitud semántica** con distancia coseno.
- Construir una **búsqueda semántica** sobre una colección de documentos.
- Demostrar **alucinaciones** y **conocimiento desactualizado**.
- Comprender los **límites de contexto** como causa de errores.

## Conexión con el proyecto ARAS

ARAS procesará documentos académicos y responderá preguntas de investigación.
La **búsqueda semántica** (embeddings) será la base de la herramienta de
recuperación; y entender las **alucinaciones** justifica por qué ARAS debe
**distinguir información encontrada de conclusiones generadas**.

## 1. Dependencias

In [ ]:
%pip install -q ollama numpy

## 2. Embeddings

Un *embedding* convierte un texto en un vector de números reales. Textos con
significado parecido producen vectores cercanos. Usamos `nomic-embed-text`,
un modelo pequeño especializado en embeddings.

In [ ]:
import ollama

resp = ollama.embed(model="nomic-embed-text", input="inteligencia artificial")
vector = resp["embeddings"][0]
print("Dimensión del vector:", len(vector))
print("Primeros 5 valores:", vector[:5])

## 3. Similitud semántica (coseno)

La **similitud coseno** mide el ángulo entre dos vectores: `1.0` = idénticos,
`0.0` = ortogonales (sin relación), `-1.0` = opuestos.

In [ ]:
import numpy as np

def coseno(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def embedding(texto):
    return ollama.embed(model="nomic-embed-text", input=texto)["embeddings"][0]

pares = [
    ("inteligencia artificial", "machine learning"),
    ("inteligencia artificial", "receta de cocina"),
    ("agente autónomo", "sistema que actúa sin supervisión humana"),
]

for t1, t2 in pares:
    v1, v2 = embedding(t1), embedding(t2)
    print(f"coseno('{t1}', '{t2}') = {coseno(v1, v2):.3f}")

## 4. Búsqueda semántica sobre una colección

Esta es la semilla de la herramienta de recuperación de ARAS: dado un conjunto
de documentos, devolvemos los más relevantes a una consulta.

In [ ]:
import ollama
import numpy as np

DOCUMENTOS = [
    "Model Context Protocol estandariza cómo los agentes acceden a herramientas y recursos.",
    "Los transformers usan atención para modelar relaciones entre tokens.",
    "La evaluación de agentes mide relevancia, uso de herramientas y trazabilidad.",
    "La temperatura controla la aleatoriedad en la generación de texto.",
    "Los embeddings permiten búsqueda semántica y agrupación de documentos.",
]

def embedding(texto):
    return ollama.embed(model="nomic-embed-text", input=texto)["embeddings"][0]

indice = np.array([embedding(d) for d in DOCUMENTOS])

def buscar(consulta, k=3):
    q = np.asarray(embedding(consulta), dtype=float)
    sims = indice @ q / (np.linalg.norm(indice, axis=1) * np.linalg.norm(q))
    orden = np.argsort(-sims)[:k]
    return [(DOCUMENTOS[i], float(sims[i])) for i in orden]

for doc, sim in buscar("¿Cómo recuperan los agentes información de fuentes externas?"):
    print(f"[{sim:.3f}] {doc}")

## 5. Alucinaciones

Un LLM **no "sabe"**, predice el texto más probable. Si le pedimos un dato que no
está en su entrenamiento, suele **inventarlo con total seguridad**. Esto es una
**alucinación**.

In [ ]:
import ollama

MODELO = "llama3.2"

prompt = (
    "Dame el título exacto, los autores y el año del artículo de la doctora "
    "María González sobre agentes autónomos aplicados a la educación superior."
)
resp = ollama.generate(model=MODELO, prompt=prompt, options={"temperature": 0.0})
print(resp["response"])

**Observa:** probablemente el modelo devolvió un título, autores y año que
*parecen* reales. No existe evidencia de ese artículo: lo ha fabricado.

**Implicación para ARAS:** nunca debemos presentar referencias inventadas como
fuentes verificadas. Toda referencia debe **comprobarse contra los datos
recuperados**.

## 6. Conocimiento desactualizado

El conocimiento de un modelo se congela en su fecha de entrenamiento. Preguntas
sobre hechos recientes producen respuestas obsoletas o inventadas.

In [ ]:
import ollama

resp = ollama.generate(
    model=MODELO,
    prompt="¿Quién ganó el último Premio Nobel de Física y en qué año?",
    options={"temperature": 0.0},
)
print(resp["response"])

## 7. Límites de contexto como fuente de error

Si un documento es más largo que la ventana de contexto, el modelo solo "ve"
una parte. Esto provoca respuestas parciales o incorrectas. Por eso ARAS
**divide los documentos en fragmentos (chunks)** antes de buscar.

In [ ]:
import ollama

# Texto largo: el modelo no puede retenerlo todo a la vez.
texto_largo = ("El capítulo tercero analiza la evaluación de sistemas multiagente. " * 400)

resp = ollama.generate(
    model=MODELO,
    prompt="Resume en una frase el tema del siguiente texto:\n\n" + texto_largo,
    options={"temperature": 0.0, "num_predict": 60},
)
print(resp["response"][:400])

## 8. Cómo mitigar alucinaciones (adelanto de ARAS)

1. **Fundamentar (grounding):** responder solo con base en documentos recuperados.
2. **Citar:** pedir al modelo que indique de qué fragmento proviene cada dato.
3. **Verificar:** comprobar las referencias contra las fuentes.
4. **Confesar ignorancia:** instruir al modelo para que diga "no lo sé" si no hay evidencia.

Estas técnicas se desarrollarán en los Módulos 2, 4 y 7.

## Resumen del laboratorio

- Generamos **embeddings** y medimos **similitud coseno**.
- Construimos **búsqueda semántica** (base de la recuperación en ARAS).
- Demostramos **alucinaciones** y **conocimiento desactualizado**.
- Relacionamos los **límites de contexto** con la necesidad de dividir documentos.